In [ ]:
!pip install langchain langchain-community duckduckgo-search gradio transformers accelerate bitsandbytes sentence-transformers
!pip install -U ddgs

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gradio as gr

In [ ]:
model_id = "tiiuae/falcon-7b-instruct"   # open + small
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
search = DuckDuckGoSearchRun()

In [ ]:
def rag_pipeline(query):
    # Step 1: Retrieve from web
    search_results = search.run(query)

    # Step 2: Build augmented prompt
    prompt = f"""
You are a helpful assistant.
Answer the following query using the web search results.

Query: {query}

Search Results:
{search_results}

Answer:
"""

    # Step 3: Generate with falcon
    response = llm(prompt)
    return response

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# 🌐 RAG with Llama 3 + Web Search (DuckDuckGo)")
    with gr.Row():
        query = gr.Textbox(label="Enter your query", placeholder="Ask me anything...")
    with gr.Row():
        output = gr.Textbox(label="Answer")
    with gr.Row():
        submit = gr.Button("Search & Generate")
    
    submit.click(rag_pipeline, inputs=query, outputs=output)

demo.launch(share=True)